# Evaluation pipeline demo


This is a short notebook showing how the evaluation pipeline works WITHOUT having to download the entire dataset and configure the container. Small subsets of each dataset is included in duckdb files in this folder.

Ensure you have Python 3 and `uv`, then just run `uv sync` from the repository root and ensure the notebook is connected to that virtual environment (likely `.venv` by default)

## Datasets
1. [Risk-Based Authentication (RBA)](https://zenodo.org/records/6782156) (Wiefling et al, 2022)
    - synthetic dataset of 31m authentication records from ~3m users with user IDs from an SSO service
    - includes user agent, IP, etc., as well as fields for (a) whether the login originates from a known attack IP address or (b) the login was flagged as an account takeover (ground truth)
    - Licensed by the authors (Wiefling et al.) under a [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

2. [FP-Stalker](https://github.com/Spirals-Team/FPStalker) (Vastel et al., 2018) 
    - 15k record sample from the AmIUnique dataset from 2015-2017. 
    - Each record contains a long term tracking cookie (tying it to other records from the same origin) alongside a full user agent string and other fingerprint attributes we do not use in our ER method (e.g., canvas).
    - Licensed by the authors (Vastel et al.) under the [GNU-AGPL 3.0 License](https://github.com/Spirals-Team/FPStalker/blob/master/LICENSE)



The datasets provided in this demo are small subsets of the above datasets just to show how this works at a high level:
1. RBA subset (3.2k rows): Selected only only "User IDs" that have at least one row flagged as "Is Account Takeover"
2. FP Stalker Subset (8.6k rows): Selected only tracking cookie IDs that were live in June 2016. Also removed some columns that aren't useful to us.


In [ ]:
import duckdb
import pandas as pd

with duckdb.connect('./trunc_rba.duckdb') as rba_conn:
    rba_df_orig = rba_conn.execute("SELECT * FROM imported_data").df()

print("Length of RBA truncated dataset:", rba_df_orig.shape[0])
display(rba_df_orig.head())

In [ ]:
with duckdb.connect('./trunc_fp_stalker.duckdb') as fp_conn:
    fp_df_orig = fp_conn.execute("SELECT * FROM imported_data").df() 

print("Length of FP Stalker truncated dataset:", fp_df_orig.shape[0])
display(fp_df_orig.head())

# Parse User Agent Strings 

Parse the user agent strings (e.g., `Mozilla/5.0 (Windows NT 6.1; WOW64; rv:41.0) Gecko/20100101 Firefox/41.0`) using the same `field_normalization` modules as the webapp, then move them into the column names expected by the `device_grouping2` module.

In [ ]:
import sys
from pathlib import Path
import tqdm

_python_core = str(Path.cwd().parent.parent / "python_core")
if _python_core not in sys.path:
    sys.path.append(_python_core)

from python_core.field_normalization.user_agent import UserAgentParser
from python_core.field_normalization.device import normalize_device_fields


ua_parser = UserAgentParser()

def normalize_data(df: pd.DataFrame, 
                   user_agent_col_name: str, 
                   timestamp_col_name: str,
                   other_cols_to_keep: list = []) -> pd.DataFrame:
    dicts = []
    for _, row in tqdm.tqdm(df.iterrows(), total=len(df)):
        d = {"user_agent_original": row.get(user_agent_col_name, "")}
        d = ua_parser.parse(d) # returns dict
        d = normalize_device_fields(d)
        d = {k: v for k, v in d.items() if k.startswith("norm__")}
        d['timestamp'] = row.get(timestamp_col_name, "")
        dicts.append(d)
    
    new_columns = pd.DataFrame(dicts)
    normalized_cols = [c for c in new_columns.columns if c.startswith("norm__")]
    
    if isinstance(other_cols_to_keep, list):
        other_cols_to_keep = [c for c in other_cols_to_keep if c in df.columns]
    else:
        other_cols_to_keep = []
    
    new_df = df[other_cols_to_keep + [user_agent_col_name, timestamp_col_name]]
    return pd.concat([new_df, new_columns[normalized_cols]], axis=1)

In [ ]:
norm_rba_df = normalize_data(rba_df_orig,
                             user_agent_col_name='User Agent String', 
                             timestamp_col_name='Login Timestamp',
                             other_cols_to_keep=['index', 'User ID', 'Login Successful', 'Is Account Takeover', 'Is Attack IP', 'Browser Name and Version', 'OS Name and Version'])                           

norm_rba_df

In [ ]:
norm_fp_df = normalize_data(fp_df_orig,
                            user_agent_col_name='userAgentHttp', 
                            timestamp_col_name='creationDate',
                            other_cols_to_keep=['id', 'counter', 'osDetailed', 'browserDetailed'])
norm_fp_df